**[EDA Visual Encyclopedia](https://patrykgolabek.dev/eda/)** — Interactive case studies, technique references, and downloadable notebooks for Exploratory Data Analysis.

# Normal Random Numbers

**NIST/SEMATECH e-Handbook of Statistical Methods, Section 1.4.2.1**

Source: [https://www.itl.nist.gov/div898/handbook/eda/section4/eda421.htm](https://www.itl.nist.gov/div898/handbook/eda/section4/eda421.htm)

## Background

### Generation

The normal random numbers used in this case study are from a Rand Corporation publication. The motivation for studying a set of normal random numbers is to illustrate the ideal case where all four underlying assumptions hold.

### Analysis Goals

1. **Location:** What is a typical value?
2. **Variation:** How spread out are the data?
3. **Distribution:** What is the shape of the distribution?
4. **Randomness:** Are the data random (no autocorrelation or trend)?
5. **Outliers:** Are there any outliers in the data?

## Environment Setup

In [ ]:
# Check dependencies and install if missing
try:
    import numpy as np
    import scipy
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install numpy scipy pandas matplotlib seaborn
    import numpy as np
    import scipy
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

print(f'NumPy {np.__version__}, SciPy {scipy.__version__}, Pandas {pd.__version__}')
print(f'Matplotlib {plt.matplotlib.__version__}, Seaborn {sns.__version__}')

In [ ]:
# Quantum Explorer dark theme for matplotlib/seaborn
# Matches the site color scheme for visual consistency

import matplotlib.pyplot as plt
import seaborn as sns

QUANTUM_COLORS = {
    'background': '#0f1117',
    'surface': '#1a1d27',
    'accent': '#e06040',
    'teal': '#00a3a3',
    'text': '#e8e8f0',
    'text_secondary': '#9ca3af',
    'border': '#2a2d3a',
    'gradient_start': '#e06040',
    'gradient_end': '#00a3a3',
}

# Color cycle for multiple series
QUANTUM_PALETTE = ['#e06040', '#00a3a3', '#f0c040', '#a080e0', '#60c0a0', '#e080a0']

plt.rcParams.update({
    'figure.facecolor': QUANTUM_COLORS['background'],
    'axes.facecolor': QUANTUM_COLORS['surface'],
    'axes.edgecolor': QUANTUM_COLORS['border'],
    'axes.labelcolor': QUANTUM_COLORS['text'],
    'axes.titlecolor': QUANTUM_COLORS['text'],
    'xtick.color': QUANTUM_COLORS['text_secondary'],
    'ytick.color': QUANTUM_COLORS['text_secondary'],
    'text.color': QUANTUM_COLORS['text'],
    'grid.color': QUANTUM_COLORS['border'],
    'grid.alpha': 0.5,
    'figure.figsize': [10, 6],
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.prop_cycle': plt.cycler('color', QUANTUM_PALETTE),
})

sns.set_theme(style='darkgrid', rc={
    'axes.facecolor': QUANTUM_COLORS['surface'],
    'figure.facecolor': QUANTUM_COLORS['background'],
    'grid.color': QUANTUM_COLORS['border'],
    'text.color': QUANTUM_COLORS['text'],
    'axes.labelcolor': QUANTUM_COLORS['text'],
    'xtick.color': QUANTUM_COLORS['text_secondary'],
    'ytick.color': QUANTUM_COLORS['text_secondary'],
})

print('Quantum Explorer theme configured.')

In [ ]:
# Additional imports for statistical analysis
from scipy import stats
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

## Data Loading

Load the **RANDN.DAT** dataset from NIST (500 observations).

In [ ]:
# Load dataset (CSV generated from NIST RANDN.DAT)
DATA_URL = 'https://raw.githubusercontent.com/PatrykQuantumNomad/PatrykQuantumNomad/main/notebooks/eda/data/normal-random-numbers.csv'
df = pd.read_csv(DATA_URL)

print(f'Loaded {len(df)} rows')
assert len(df) == 500, f'Expected 500 rows, got {len(df)}'

In [ ]:
# Preview the first few rows
print(df.head(10))
print()
print(f'Shape: {df.shape}')
print(f'Data types:\n{df.dtypes}')

## Summary Statistics

Compute key descriptive statistics for the **Y** variable.

In [ ]:
# Summary statistics for Y
y = df['Y']

summary = pd.DataFrame({
    'Statistic': ['Mean', 'Std Dev', 'Median', 'Min', 'Max',
                  'Skewness', 'Kurtosis', 'N'],
    'Value': [
        y.mean(),
        y.std(ddof=1),
        y.median(),
        y.min(),
        y.max(),
        y.skew(),
        y.kurtosis(),
        len(y),
    ]
})

print(summary.to_string(index=False))

## 4-Plot Analysis

The 4-plot is a collection of four graphical EDA techniques whose purpose is to test 
the assumptions that underlie most measurement processes:

1. **Run Sequence Plot** (upper left) -- tests fixed location and variation
2. **Lag Plot** (upper right) -- tests randomness
3. **Histogram** (lower left) -- tests distributional assumptions
4. **Normal Probability Plot** (lower right) -- tests normality

In [ ]:
# 4-Plot: 4-Plot of Normal Random Numbers
y = df['Y'].values

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('4-Plot of Normal Random Numbers',
             fontsize=16, color=QUANTUM_COLORS['text'], y=0.98)

# 1. Run Sequence Plot (upper left)
ax = axes[0, 0]
ax.plot(range(len(y)), y, '.', color=QUANTUM_COLORS['accent'],
        markersize=3, alpha=0.7)
ax.axhline(y.mean(), color=QUANTUM_COLORS['teal'],
           linestyle='--', linewidth=1, label=f'Mean = {y.mean():.4f}')
ax.set_xlabel('Observation Number')
ax.set_ylabel('Y')
ax.set_title('Run Sequence Plot')
ax.legend(fontsize=9)

# 2. Lag Plot (upper right)
ax = axes[0, 1]
ax.scatter(y[:-1], y[1:], c=QUANTUM_COLORS['accent'],
           s=5, alpha=0.5)
ax.set_xlabel('Y(i)')
ax.set_ylabel('Y(i+1)')
ax.set_title('Lag Plot (lag=1)')

# 3. Histogram (lower left)
ax = axes[1, 0]
ax.hist(y, bins='auto', color=QUANTUM_COLORS['accent'],
        edgecolor=QUANTUM_COLORS['border'], alpha=0.8)
ax.set_xlabel('Y')
ax.set_ylabel('Frequency')
ax.set_title('Histogram')

# 4. Normal Probability Plot (lower right)
ax = axes[1, 1]
stats.probplot(y, dist='norm', plot=ax)
ax.get_lines()[0].set_color(QUANTUM_COLORS['accent'])
ax.get_lines()[1].set_color(QUANTUM_COLORS['teal'])
ax.set_title('Normal Probability Plot')

plt.tight_layout()
plt.show()

## Individual Plots

Full-size versions of each plot for detailed examination.

### Run Sequence Plot

Tests fixed location and fixed variation assumptions.

In [ ]:
# Run Sequence Plot of Normal Random Numbers
y = df['Y'].values

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(range(len(y)), y, '.', color=QUANTUM_COLORS['accent'],
        markersize=4, alpha=0.7)
ax.axhline(y.mean(), color=QUANTUM_COLORS['teal'],
           linestyle='--', linewidth=1.5, label=f'Mean = {y.mean():.4f}')
ax.axhline(y.mean() + y.std(), color=QUANTUM_COLORS['text_secondary'],
           linestyle=':', linewidth=1, alpha=0.5, label=f'+1 SD')
ax.axhline(y.mean() - y.std(), color=QUANTUM_COLORS['text_secondary'],
           linestyle=':', linewidth=1, alpha=0.5, label=f'-1 SD')
ax.set_xlabel('Observation Number')
ax.set_ylabel('Y')
ax.set_title('Run Sequence Plot of Normal Random Numbers')
ax.legend()
plt.tight_layout()
plt.show()

### Lag Plot

Tests randomness assumption by plotting Y(i) vs Y(i-1).

In [ ]:
# Lag Plot of Normal Random Numbers
y = df['Y'].values

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y[:-1], y[1:], c=QUANTUM_COLORS['accent'],
           s=8, alpha=0.5, edgecolors='none')

# Add diagonal reference line
lims = [min(y.min(), y.min()), max(y.max(), y.max())]
ax.plot(lims, lims, '--', color=QUANTUM_COLORS['teal'],
        linewidth=1, alpha=0.5)
ax.set_xlabel('Y(i)')
ax.set_ylabel('Y(i+1)')
ax.set_title('Lag Plot of Normal Random Numbers')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### Histogram

Tests distributional assumptions by showing the frequency distribution.

In [ ]:
# Histogram of Normal Random Numbers
y = df['Y'].values

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(y, bins='auto', color=QUANTUM_COLORS['accent'],
        edgecolor=QUANTUM_COLORS['border'], alpha=0.8, density=True)

# Overlay normal curve
x_range = np.linspace(y.min(), y.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, y.mean(), y.std()),
        color=QUANTUM_COLORS['teal'], linewidth=2, label='Normal fit')

ax.set_xlabel('Y')
ax.set_ylabel('Density')
ax.set_title('Histogram of Normal Random Numbers')
ax.legend()
plt.tight_layout()
plt.show()

### Normal Probability Plot

Tests normality assumption. Points close to the line indicate normal distribution.

In [ ]:
# Normal Probability Plot of Normal Random Numbers
y = df['Y'].values

fig, ax = plt.subplots(figsize=(8, 8))
res = stats.probplot(y, dist='norm', plot=ax)

ax.get_lines()[0].set_color(QUANTUM_COLORS['accent'])
ax.get_lines()[0].set_markersize(4)
ax.get_lines()[1].set_color(QUANTUM_COLORS['teal'])
ax.get_lines()[1].set_linewidth(2)

ax.set_title('Normal Probability Plot of Normal Random Numbers')
plt.tight_layout()
plt.show()

## Hypothesis Tests

We apply five categories of hypothesis tests to evaluate the assumptions
underlying a standard univariate EDA analysis.

### 1. Location Test

Test whether the data have a significant linear trend (drift in location)
using ordinary least-squares regression of the response on run order.

In [ ]:
# Location test: linear regression of Y on run order
y = df['Y'].values
x = np.arange(1, len(y) + 1)

from scipy.stats import linregress

slope, intercept, r_value, p_value, std_err = linregress(x, y)

print('Location Test (Linear Regression)')
print('=' * 40)
print(f'Slope:     {slope:.6f}')
print(f'Intercept: {intercept:.6f}')
print(f'R-squared: {r_value**2:.6f}')
print(f'p-value:   {p_value:.6f}')
print(f'Std Error: {std_err:.6f}')
print()
alpha = 0.05
if p_value < alpha:
    print(f'Result: REJECT H0 (p={p_value:.4f} < {alpha}) -- significant trend detected')
else:
    print(f'Result: FAIL TO REJECT H0 (p={p_value:.4f} >= {alpha}) -- no significant trend')

### 2. Variation Test

Bartlett's test for equality of variances across groups.
We split the data into 4 equal groups and test whether their variances differ.

In [ ]:
# Variation test: Bartlett's test on 4 equal groups
y = df['Y'].values
n = len(y)
q = n // 4
groups = [y[i*q:(i+1)*q] for i in range(4)]
# Include remainder in last group
if n % 4 != 0:
    groups[-1] = y[3*q:]

from scipy.stats import bartlett

stat, p_value = bartlett(*groups)

print("Bartlett's Test for Equal Variances")
print('=' * 40)
print(f'Test statistic: {stat:.4f}')
print(f'p-value:        {p_value:.6f}')
alpha = 0.05
if p_value < alpha:
    print(f'Result: REJECT H0 (p={p_value:.4f} < {alpha}) -- variances differ')
else:
    print(f'Result: FAIL TO REJECT H0 (p={p_value:.4f} >= {alpha}) -- variances are equal')

### 3. Randomness Tests

Two tests for randomness:
- **Runs test** (manual implementation, no external dependencies)
- **Lag-1 autocorrelation** with critical value 2/sqrt(N)

In [ ]:
# Manual runs test implementation
def runs_test(data):
    """
    Wald-Wolfowitz runs test for randomness.
    Returns (n_runs, expected_runs, z_statistic, p_value).
    """
    median = np.median(data)
    binary = [1 if x >= median else 0 for x in data]
    n1 = sum(binary)
    n2 = len(binary) - n1

    # Count runs
    runs = 1
    for i in range(1, len(binary)):
        if binary[i] != binary[i - 1]:
            runs += 1

    # Expected runs and standard deviation
    expected = (2 * n1 * n2) / (n1 + n2) + 1
    std = np.sqrt((2 * n1 * n2 * (2 * n1 * n2 - n1 - n2)) /
                  ((n1 + n2)**2 * (n1 + n2 - 1)))

    z = (runs - expected) / std if std > 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return runs, expected, z, p

y = df['Y'].values
n_runs, exp_runs, z_stat, p_val = runs_test(y)

print('Runs Test for Randomness')
print('=' * 40)
print(f'Observed runs: {n_runs}')
print(f'Expected runs: {exp_runs:.1f}')
print(f'Z-statistic:   {z_stat:.4f}')
print(f'p-value:       {p_val:.6f}')
alpha = 0.05
if p_val < alpha:
    print(f'Result: REJECT H0 (p={p_val:.4f} < {alpha}) -- data are NOT random')
else:
    print(f'Result: FAIL TO REJECT H0 (p={p_val:.4f} >= {alpha}) -- data appear random')

In [ ]:
# Lag-1 autocorrelation with critical value 2/sqrt(N)
y = df['Y'].values
N = len(y)
y_mean = np.mean(y)

# Compute lag-1 autocorrelation coefficient
numerator = np.sum((y[1:] - y_mean) * (y[:-1] - y_mean))
denominator = np.sum((y - y_mean) ** 2)
r1 = numerator / denominator

# Critical value: 2/sqrt(N)
critical = 2 / np.sqrt(N)

print('Lag-1 Autocorrelation Test')
print('=' * 40)
print(f'Lag-1 autocorrelation: {r1:.6f}')
print(f'Critical value (+/-): {critical:.6f}')
print(f'Sample size N:        {N}')
print()
if abs(r1) > critical:
    print(f'Result: |r1| = {abs(r1):.4f} > {critical:.4f} -- significant autocorrelation detected')
else:
    print(f'Result: |r1| = {abs(r1):.4f} <= {critical:.4f} -- no significant autocorrelation')

### 4. Distribution Test (Anderson-Darling)

The Anderson-Darling test checks whether the data follow a specified distribution.
We test for normality.

In [ ]:
# Anderson-Darling test for normality
y = df['Y'].values

from scipy.stats import anderson

ad_result = anderson(y, dist="norm")

print('Anderson-Darling Test for Normality')
print('=' * 40)
print(f'Statistic: {ad_result.statistic:.4f}')
print('Critical values:')
for sl, cv in zip(ad_result.significance_level, ad_result.critical_values):
    flag = ' ***' if ad_result.statistic > cv else ''
    print(f'  {sl:5.1f}%: {cv:.4f}{flag}')
print()
if ad_result.statistic > ad_result.critical_values[2]:
    print('Result: REJECT normality at 5% significance level')
else:
    print('Result: FAIL TO REJECT normality at 5% significance level')

### 5. Outlier Test (Grubbs)

Grubbs' test identifies single outliers in a univariate dataset.

In [ ]:
# Grubbs' test for outliers (manual implementation)
def grubbs_test(data, alpha=0.05):
    """
    Grubbs' test for a single outlier.',
    Returns (G_statistic, critical_value, is_outlier, suspect_value).
    """
    n = len(data)
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    abs_devs = np.abs(data - mean)
    max_idx = np.argmax(abs_devs)
    G = abs_devs[max_idx] / std

    # Critical value from t-distribution
    from scipy.stats import t as t_dist
    t_crit = t_dist.ppf(1 - alpha / (2 * n), n - 2)
    G_crit = ((n - 1) / np.sqrt(n)) * np.sqrt(t_crit**2 / (n - 2 + t_crit**2))

    return G, G_crit, G > G_crit, data[max_idx]

y = df['Y'].values
G, G_crit, is_outlier, suspect = grubbs_test(y)

print("Grubbs' Test for Outliers")
print('=' * 40)
print(f'G statistic:    {G:.4f}')
print(f'Critical value: {G_crit:.4f}')
print(f'Suspect value:  {suspect:.4f}')
if is_outlier:
    print(f'Result: {suspect:.4f} IS an outlier (G={G:.4f} > {G_crit:.4f})')
else:
    print(f'Result: No outlier detected (G={G:.4f} <= {G_crit:.4f})')

## Test Summary

Summary of all hypothesis test results for this case study.

In [ ]:
# Test Summary Table
print('=' * 70)
print('Test Summary')
print('=' * 70)

summary_data = [
    ['Location', f'{slope:.6f}', f'{p_value:.6f}', 'Trend detected' if p_value < alpha else 'No trend'],
    ['Variation', f'{stat:.4f}', f'{p_value:.6f}', 'Variances differ' if p_value < alpha else 'Variances equal'],
    ['Randomness (Runs)', f'{z_stat:.4f}', f'{p_val:.6f}', 'Not random' if p_val < alpha else 'Random'],
    ['Randomness (Lag-1)', f'{r1:.6f}', f'{critical:.6f}', 'Autocorrelated' if abs(r1) > critical else 'No autocorrelation'],
        ['Distribution', f'{ad_result.statistic:.4f}', f'{ad_result.critical_values[2]:.4f}', 'Reject' if ad_result.statistic > ad_result.critical_values[2] else 'Fail to reject'],
        ['Outlier', f'{G:.4f}', f'{G_crit:.4f}', 'Outlier detected' if is_outlier else 'No outlier'],
]

headers = ['Test', 'Statistic', 'Threshold', 'Conclusion']

# Print formatted table
print(f"{'Test':<22} {'Statistic':>12} {'Threshold':>12} {'Conclusion'}")
print('-' * 70)
for row in summary_data:
    print(f'{row[0]:<22} {row[1]:>12} {row[2]:>12} {row[3]}')

## Interpretation

### Key Findings for Normal Random Numbers

The normal random numbers dataset is the **baseline case study** where all
standard EDA assumptions are expected to be satisfied:

1. **Location:** The mean is close to the theoretical value of 0, and no
   significant trend was detected. The location is fixed.
2. **Variation:** The standard deviation is close to the theoretical value of 1.
   Bartlett's test confirms equal variances across groups.
3. **Randomness:** Both the runs test and lag-1 autocorrelation confirm that
   the data are random with no serial dependence.
4. **Distribution:** The Anderson-Darling test confirms normality. The probability
   plot shows points closely following the theoretical line.
5. **Outliers:** No significant outliers detected by Grubbs' test.

All assumptions are satisfied, confirming that the underlying random number
generator produces data consistent with a N(0,1) process. The sample mean
and sample standard deviation are sufficient statistics, and standard
confidence intervals and hypothesis tests can be applied without concern.

This case study serves as a **pass/fail reference**: if your EDA pipeline
produces the same conclusions on this dataset, your implementation is correct.

## Conclusions

### Key Findings

1. All five EDA assumptions (fixed location, fixed variation, randomness, normality, no outliers) are satisfied.
2. The dataset confirms that the random number generator produces N(0,1) distributed data.
3. Sample mean and standard deviation are reliable point estimates.

### Next Steps

1. Use this dataset as a baseline reference for validating EDA pipelines.
2. Standard parametric methods (t-tests, F-tests) can be applied directly.
3. No further modeling is needed.

---

### References

- NIST/SEMATECH e-Handbook of Statistical Methods: [https://www.itl.nist.gov/div898/handbook/eda/section4/eda421.htm](https://www.itl.nist.gov/div898/handbook/eda/section4/eda421.htm)
- Section 1.4.2.1: Normal Random Numbers